# Pune Property Price Analysis🏠🏘️

## 03. Feature Engineering Notebook

 To transform raw property data into a structured and model-ready format by handling missing values, encoding categorical features, and scaling numerical variables to improve model performance.

## Unfurnished & Furnished Property Price Prediction

# 1. Loading the Libraries & Models

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import json
import joblib
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV


# Models
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score


# 2. Data Analysis

In [2]:
# Loading the Cleaned data

data = pd.read_csv("/content/cleaned_pune_property_ds.csv")

In [3]:
# First few Rows

data.head()

,balconies,bathroom,facing,locality,opensides,possesiondate,price,totalfloor,no. of additional rooms,studyroom,...,Wardrobe,TV,Refrigerator,Sofa,Washing Machine,Wifi,Microwave,Dining Table,Gas connection,BED
0,2.0,3.0,West,Deccan Gymkhana,2.0,NaN,27000000,6.0,NaN,NaN,...,No,No,No,No,No,No,No,No,Yes,No
1,NaN,2.0,NorthEast,Kharadi,NaN,NaN,7610050,NaN,NaN,NaN,...,No,No,No,No,No,No,No,No,No,No
2,NaN,1.0,NorthEast,Sus,NaN,NaN,4100000,NaN,NaN,NaN,...,No,No,No,No,No,No,No,No,No,No
3,NaN,2.0,NorthEast,Bavdhan,NaN,NaN,5736000,NaN,NaN,NaN,...,No,No,No,No,No,No,No,No,No,No
4,1.0,3.0,West,Deccan Gymkhana,2.0,NaN,29002000,9.0,NaN,NaN,...,No,No,No,No,No,No,No,No,No,No


In [4]:
# Columns

data.columns

Index(['balconies', 'bathroom', 'facing', 'locality', 'opensides',
       'possesiondate', 'price', 'totalfloor', 'no. of additional rooms',
       'studyroom', 'poojaroom', 'servantroom', 'age of house', 'total area',
       'total rooms', 'apartment', 'residential', 'penthouse', 'independent',
       'villa', 'carpet area', 'floors', 'house type', 'road view',
       'garden view', 'phool view', 'corner view', 'co-operative society',
       'freehold property', 'power of attorney', 'leasehold', 'possesion year',
       'possesion month', 'road feet', 'ready to move', 'unfurnished',
       'under construction', 'semi furnished', 'furnished', 'Lift Available',
       'Car Parking', 'Power Backup', '24 X 7 Security',
       'Children's play area', 'Vaastu Compliant', 'Club House', 'Gymnasium',
       'Swimming Pool', 'Sports Facility', 'Indoor Games', 'Jogging Track',
       'Maintenance Staff', 'Intercom', 'Golf Course', 'Cafeteria',
       'Rain Water Harvesting', 'Staff Quarter', 'Mu

In [5]:
# Data Information

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21912 entries, 0 to 21911
Data columns (total 74 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   balconies                12017 non-null  float64
 1   bathroom                 20433 non-null  float64
 2   facing                   18191 non-null  object 
 3   locality                 21912 non-null  object 
 4   opensides                4756 non-null   float64
 5   possesiondate            1909 non-null   object 
 6   price                    21912 non-null  int64  
 7   totalfloor               13252 non-null  float64
 8   no. of additional rooms  5123 non-null   float64
 9   studyroom                5123 non-null   object 
 10  poojaroom                5123 non-null   object 
 11  servantroom              5123 non-null   object 
 12  age of house             16832 non-null  float64
 13  total area               21904 non-null  float64
 14  total rooms           

# 3. Checking Null Values

In [6]:
data.isnull().sum()

,0
balconies,9895
bathroom,1479
facing,3721
locality,0
opensides,17156
...,...
Wifi,1
Microwave,1
Dining Table,1
Gas connection,1


# 4. Unfurnished Property Price Prediction

* Since users cannot provide all property details, a minimal set of primary features was selected from a user’s point of view.  
* Only features that are commonly known while searching for unfurnished flats were retained.

## 4.1 Selecting features for Unfurnished Flats

In [7]:
unfurnished = data[["balconies", "bathroom", "price","house type",
     "no. of additional rooms", "total area",
     "total rooms", "Car Parking", "Power Backup"]]

In [8]:
un = unfurnished
un

,balconies,bathroom,price,house type,no. of additional rooms,total area,total rooms,Car Parking,Power Backup
0,2.0,3.0,27000000,New,NaN,1611.0,5.0,Yes,Yes
1,NaN,2.0,7610050,New,NaN,1279.0,4.0,Yes,Yes
2,NaN,1.0,4100000,New,NaN,670.0,3.0,Yes,Yes
3,NaN,2.0,5736000,New,NaN,956.0,4.0,Yes,Yes
4,1.0,3.0,29002000,New,NaN,1706.0,5.0,Yes,Yes
...,...,...,...,...,...,...,...,...,...
21907,NaN,3.0,10400000,New,NaN,1799.0,5.0,Yes,Yes
21908,NaN,2.0,15000000,New,NaN,830.0,4.0,Yes,Yes
21909,NaN,2.0,6800000,New,NaN,1374.0,5.0,Yes,Yes
21910,NaN,2.0,5000000,New,NaN,1225.0,4.0,Yes,Yes


## 4.2 Checking Null values in Unfurnished data

In [9]:
un.isnull().sum()

,0
balconies,9895
bathroom,1479
price,0
house type,0
no. of additional rooms,16789
total area,8
total rooms,483
Car Parking,0
Power Backup,0


## 4.3 Dividing the Data into x and y
The target variable for prediction is:
- `price`

In [10]:
x = un.drop("price", axis = 1)
y = un["price"]

## 4.4 Dividing the Data into training and testing phase

In [11]:
# Train Test Split

xtrain, xtest, ytrain, ytest = train_test_split(x,y,test_size = 0.20, random_state=42)

In [12]:
# Checking Rows and Columns

xtrain.shape, xtest.shape, ytrain.shape, ytest.shape

((17529, 8), (4383, 8), (17529,), (4383,))

In [13]:
xtrain.head()

,balconies,bathroom,house type,no. of additional rooms,total area,total rooms,Car Parking,Power Backup
21540,NaN,3.0,Old,NaN,1930.0,5.0,Yes,Yes
4073,1.0,2.0,Old,NaN,1450.0,4.0,Yes,Yes
9994,1.0,2.0,Old,1.0,1265.0,4.0,Yes,Yes
11066,NaN,2.0,Old,NaN,1120.0,4.0,Yes,Yes
18771,1.0,1.0,Old,NaN,630.0,3.0,No,No


In [14]:
ytrain.head()

,price
21540,26000000
4073,7000000
9994,10000000
11066,5500000
18771,8800000


## 4.5 Pipelines and Column Transformer

- Pipeline : Sequentially applies preprocessing steps to specific features.

- ColumnTransformer : APplies different preprocessing pipeline to different columns

In [15]:
# Filling missing values with 0 for balconies and additional rooms
missing_constant_0 = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy = "constant", fill_value =0))
    ]
)

# Filling missing values with median for bathroom and total rooms
missing_median = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

# Encode house type as ordinal (New=0, Old=1)
categ_ordinal = Pipeline(
    steps = [
        ("encode", OrdinalEncoder(categories=[["New", "Old"]]))
    ]
)

# Filling missing with "Yes" and encode Car Parking & Power Backup (No=0, Yes=1)
missing_mode_ordinal = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy = "constant" , fill_value = "Yes")),
        ("encode", OrdinalEncoder(categories = [["No", "Yes"], ["No", "Yes"]]))
    ]
)

# Impute median and scale total area using MinMaxScaler
area_min_max = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy = "median")),
        ("scaler", MinMaxScaler())
    ]
)



### Combining all preprocessing pipelines using ColumnTransformer



In [16]:
transformer = ColumnTransformer(
    transformers = [
        ("pipe 1", missing_constant_0, ["balconies", "no. of additional rooms"]),
        ("pipe 2", missing_median, ["bathroom", "total rooms"]),
        ("pipe 3", categ_ordinal, ["house type"]),
        ("pipe 4", missing_mode_ordinal, ["Car Parking", "Power Backup"]),
        ("pipe 5", area_min_max,["total area"])
    ],
    remainder = "passthrough"
)

## 4.6 Transformer fit & transform on xtrain and xtest

In [17]:
# fit on xtrain
transformer.fit(xtrain)

ColumnTransformer(remainder='passthrough',
                  transformers=[('pipe 1',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value=0,
                                                                strategy='constant'))]),
                                 ['balconies', 'no. of additional rooms']),
                                ('pipe 2',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['bathroom', 'total rooms']),
                                ('pipe 3',
                                 Pipeline(steps=[('encode',
                                                  OrdinalEncoder(categories=[['New',
                                                                              'Old']]))]),
                                 ['house type']),
                                ('pipe 4',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='Yes',
                                                                strategy='constant')),
                                                 ('encode',
                                                  OrdinalEncoder(categories=[['No',
                                                                              'Yes'],
                                                                             ['No',
                                                                              'Yes']]))]),
                                 ['Car Parking', 'Power Backup']),
                                ('pipe 5',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', MinMaxScaler())]),
                                 ['total area'])])

In [18]:
# Transform on xtrain and xtest
xtrain_t = transformer.transform(xtrain)
xtest_t = transformer.transform(xtest)

## 4.7 Transforming the ytrain and ytest


In [19]:
ytrain.head()

,price
21540,26000000
4073,7000000
9994,10000000
11066,5500000
18771,8800000


### Prices conversion into log1p
- log1p is preferred over log because it safely handles zero values and provides better numerical stability without affecting large values

In [20]:
#log1p
ytrain_t = np.log1p(ytrain)
ytest_t = np.log1p(ytest)

## 4.8 Loading on ML Models

In [21]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest" : RandomForestRegressor(random_state= 42),
    "Gradient Boosting" : GradientBoostingRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state = 42, n_estimators = 100, learning_rate = 0.1, max_depth = 6, verbosity = 0)
}

## 4.9 Training the models on xtrain and ytrain

In [22]:
results = []

for name, model in models.items():

  # Train
  model.fit(xtrain_t, ytrain_t)

  # Predict
  y_pred = model.predict(xtest_t)

  # Metrics
  mae = mean_absolute_error(ytest_t, y_pred)
  r2 = r2_score(ytest_t, y_pred)

  # Storing the Results
  results.append({
      "Model" :name,
      "MAE" : mae,
      "R2 Score": r2
  })

  print("="*60)
  print(f"\nModel : {name}\n")
  print(f"MAE : {mae}")
  print(f"R2 Score: {r2*100:.2f}%")
  print("="*60)



Model : Linear Regression

MAE : 0.3659552969684183
R2 Score: 68.19%

Model : Random Forest

MAE : 0.25395202083459056
R2 Score: 82.11%

Model : Gradient Boosting

MAE : 0.2850469662217089
R2 Score: 80.56%

Model : XGBoost

MAE : 0.26778063896732346
R2 Score: 82.45%


## 4.10 Comparision of Loaded Model

- MAE directly tells average price mistake.

In [23]:
# Results dataframe

results_df = pd.DataFrame(results)
results_df.sort_values(by = "R2 Score", ascending = False)

,Model,MAE,R2 Score
3,XGBoost,0.267781,0.824464
1,Random Forest,0.253952,0.821059
2,Gradient Boosting,0.285047,0.805624
0,Linear Regression,0.365955,0.681853


## 4.11 Hyperarameter Tunning
- Random Forest

- XGBoost

### 1. Random Forest

In [24]:
# Parameter Grid

rf_params = {
    "n_estimators": [100,200,300,500], # number of decision trees in the forest
    "max_depth" : [5, 10, 20, 30], #  maximum depth of each decision tree
    "min_samples_split" : [2,5, 10], # minimum number of samples required to split an internal node
    "min_samples_leaf" : [1,2,4], # minimum number of samples required at a leaf node
    "max_features": ["sqrt", "log2"] #features considered at each split
}

In [25]:
# Randomized Search

rf = RandomForestRegressor(random_state = 42)

rf_random = RandomizedSearchCV(
    estimator = rf,
    param_distributions = rf_params,
    n_iter = 20,
    cv = 5,
    scoring = "neg_mean_absolute_error",
    random_state = 42,
    n_jobs = -1
)

rf_random.fit(xtrain_t, ytrain_t)

RandomizedSearchCV(cv=5, estimator=RandomForestRegressor(random_state=42),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'max_depth': [5, 10, 20, 30],
                                        'max_features': ['sqrt', 'log2'],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [100, 200, 300, 500]},
                   random_state=42, scoring='neg_mean_absolute_error')

In [26]:
best_rf = rf_random.best_estimator_

y_pred_rf = best_rf.predict(xtest_t)

mae_unfurnished=  mean_absolute_error(ytest_t, y_pred_rf)

print("Best RF MAE : ",mae_unfurnished)
print("Best RF R2 Score: ", r2_score(ytest_t, y_pred_rf))

Best RF MAE :  0.25786260294114793
Best RF R2 Score:  0.831816976635224


### 2. XGBoost

In [27]:
xgb_params = {
    "n_estimators": [100,200,300,500],
    "max_depth" : [3, 5, 7, 10],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree" : [0.7, 0.8, 0.9, 1.0]
}

In [28]:
xgb = XGBRegressor(random_state = 42, verbosity = 0)

xgb_random = RandomizedSearchCV(
    estimator = xgb,
    param_distributions = xgb_params,
    n_iter = 20,
    cv = 5,
    scoring = "neg_mean_absolute_error",
    random_state = 42,
    n_jobs = -1
)

xgb_random.fit(xtrain_t, ytrain_t)

RandomizedSearchCV(cv=5,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          feature_weights=None, gamma=None,
                                          grow_policy=None,
                                          importance_type=None,
                                          interaction_constraint...
                                          monotone_constraints=None,
                                          multi_strategy=None,
                                          n_estimators=None, n_jobs=None,
                                          num_parallel_tree=None, ...),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.7, 0.8, 0.9,
                                                             1.0],
                                        'learning_rate': [0.01, 0.05, 0.1, 0.2],
                                        'max_depth': [3, 5, 7, 10],
                                        'n_estimators': [100, 200, 300, 500],
                                        'subsample': [0.7, 0.8, 0.9, 1.0]},
                   random_state=42, scoring='neg_mean_absolute_error')

In [29]:
best_xgb = xgb_random.best_estimator_

y_pred_xgb = best_xgb.predict(xtest_t)

mae_unfurnished=  mean_absolute_error(ytest_t, y_pred_xgb)

print("Best RF MAE : ",mae_unfurnished)
print("Best XGB R2: ", r2_score(ytest_t, y_pred_xgb))

Best RF MAE :  0.2605488334981219
Best XGB R2:  0.8297903274221853


## 4.12 Selecting the Best Performance Model

- **Random Forest** selected as final model due to lowest MAE and strong R² performance compared to other regressors.

## 4.13 Saving Tunned Model

In [30]:
# Saving the models

joblib.dump(transformer, "unfurnished_transformer.pkl")
joblib.dump(best_rf, "rf_unfurnished_tunned.pkl")

['rf_unfurnished_tunned.pkl']

## 4.14 Saving the mae in json

In [31]:
metrics = {"mae": mae_unfurnished}

with open("rf_unfurnished_mae.json", "w") as f:
  json.dump(metrics, f)

## 4.15 Final Production Pipeline

In [32]:
final_pipeline = Pipeline(
    steps=[
        ("preprcessing", transformer),
        ("model", best_rf)
    ]
)

## 4.16 Saving the Final Production Model

In [33]:
joblib.dump(final_pipeline,"rf_unfurnished_final_production_model.pkl" )

['rf_unfurnished_final_production_model.pkl']

## 4.17 Testing the model

In [34]:
# User Input

user = {
    "balconies" : 1.0,
    "bathroom" : 2.0,
    "house type" : "New",
    "no. of additional rooms" : 0.0,
    "total area" : 1200.0,
    "total rooms" : 3.0,
    "Car Parking" : "Yes",
    "Power Backup" : "Yes"
}

In [35]:
# Converting to Data Frame

user_input = pd.DataFrame([user])
user_input

,balconies,bathroom,house type,no. of additional rooms,total area,total rooms,Car Parking,Power Backup
0,1.0,2.0,New,0.0,1200.0,3.0,Yes,Yes


In [36]:
# Loading the Saved Final production Model

model = joblib.load("rf_unfurnished_final_production_model.pkl")

In [37]:
# Average Price Predicted

print("The Predicted Price: ",np.exp(model.predict(user_input)))

The Predicted Price:  [6492520.43313401]


# 5. Furnished Property Price Prediction

- Building and evaluating regression models to predict prices of furnished properties.

- Only features that are commonly known while searching for furnished flats were retained.

## 5.1 Selecting Features for Furnished Flats

In [38]:
# Furnished

furnished = data[["balconies", "bathroom", "house type","price", "no. of additional rooms",
                  "total area", "total rooms", "Car Parking", "Power Backup", "AC", "TV",
                  "Refrigerator", "Sofa", "Wardrobe" ,"Washing Machine", "Gas connection", "BED"]]

In [39]:
fn = furnished
fn

,balconies,bathroom,house type,price,no. of additional rooms,total area,total rooms,Car Parking,Power Backup,AC,TV,Refrigerator,Sofa,Wardrobe,Washing Machine,Gas connection,BED
0,2.0,3.0,New,27000000,NaN,1611.0,5.0,Yes,Yes,No,No,No,No,No,No,Yes,No
1,NaN,2.0,New,7610050,NaN,1279.0,4.0,Yes,Yes,No,No,No,No,No,No,No,No
2,NaN,1.0,New,4100000,NaN,670.0,3.0,Yes,Yes,No,No,No,No,No,No,No,No
3,NaN,2.0,New,5736000,NaN,956.0,4.0,Yes,Yes,No,No,No,No,No,No,No,No
4,1.0,3.0,New,29002000,NaN,1706.0,5.0,Yes,Yes,No,No,No,No,No,No,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21907,NaN,3.0,New,10400000,NaN,1799.0,5.0,Yes,Yes,No,No,No,No,No,No,No,No
21908,NaN,2.0,New,15000000,NaN,830.0,4.0,Yes,Yes,No,No,No,No,No,No,No,No
21909,NaN,2.0,New,6800000,NaN,1374.0,5.0,Yes,Yes,No,No,No,No,No,No,No,No
21910,NaN,2.0,New,5000000,NaN,1225.0,4.0,Yes,Yes,No,No,No,No,No,No,No,No


## 5.2 Checking Null values in Furnished data

In [40]:
fn.isnull().sum()

,0
balconies,9895
bathroom,1479
house type,0
price,0
no. of additional rooms,16789
total area,8
total rooms,483
Car Parking,0
Power Backup,0
AC,1


## 5.3 Dividing the furnished data into x & y

In [41]:
x = fn.drop("price", axis = 1)
y = fn["price"]

## 5.4 Dividing the Data into Training and Testing Phase

In [42]:
# Train Test Split

xtrain, xtest, ytrain, ytest = train_test_split(x,y,test_size = 0.2, random_state=42)

In [43]:
# Checking Rows and Columns

xtrain.shape, xtest.shape, ytrain.shape, ytest.shape

((17529, 16), (4383, 16), (17529,), (4383,))

## 5.5 Pipelines and Column Transformer

In [44]:
missing_constant_0 = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy = "constant", fill_value =0))
    ]
)

missing_median = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categ_ordinal = Pipeline(
    steps = [
        ("encode", OrdinalEncoder(categories=[["New", "Old"]]))
    ]
)

missing_mode_ordinal = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy = "constant" , fill_value = "Yes")),
        ("encode", OrdinalEncoder())
    ]
)

area_min_max = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy = "median")),
        ("scaler", MinMaxScaler())
    ]
)



### ColumnTransformer for combning pipelines

In [45]:
transformer = ColumnTransformer(
    transformers = [
        ("pipe 1", missing_constant_0, ["balconies", "no. of additional rooms"]),
        ("pipe 2", missing_median, ["bathroom", "total rooms"]),
        ("pipe 3", categ_ordinal, ["house type"]),
        ("pipe 4", missing_mode_ordinal, [ "Car Parking", "Power Backup", "AC", "TV",
                  "Refrigerator", "Sofa", "Wardrobe" ,"Washing Machine", "Gas connection", "BED"]),
        ("pipe 5", area_min_max,["total area"])
    ],
    remainder = "passthrough"
)

## 5.6 Transformer fit & transform on xtrain and xtest

In [46]:
# fit on xtrain

transformer.fit(xtrain)

ColumnTransformer(remainder='passthrough',
                  transformers=[('pipe 1',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value=0,
                                                                strategy='constant'))]),
                                 ['balconies', 'no. of additional rooms']),
                                ('pipe 2',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['bathroom', 'total rooms']),
                                ('pipe 3',
                                 Pipeline(steps=[('encode',
                                                  OrdinalEncoder(categories=[['New',
                                                                              '...
                                 ['house type']),
                                ('pipe 4',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='Yes',
                                                                strategy='constant')),
                                                 ('encode', OrdinalEncoder())]),
                                 ['Car Parking', 'Power Backup', 'AC', 'TV',
                                  'Refrigerator', 'Sofa', 'Wardrobe',
                                  'Washing Machine', 'Gas connection', 'BED']),
                                ('pipe 5',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', MinMaxScaler())]),
                                 ['total area'])])

In [47]:
# Transform on xtrain and xtest

xtrain_t= transformer.transform(xtrain)
xtest_t = transformer.transform(xtest)

In [48]:
pd.DataFrame(xtrain_t).isnull().sum()

,0
0,0
1,0
2,0
3,0
4,0
5,0
6,0
7,0
8,0
9,0


In [49]:
xtrain_t

array([[0.        , 0.        , 3.        , ..., 0.        , 0.        ,
        0.02974102],
       [1.        , 0.        , 2.        , ..., 0.        , 0.        ,
        0.02172097],
       [1.        , 1.        , 2.        , ..., 0.        , 0.        ,
        0.01862991],
       ...,
       [1.        , 1.        , 2.        , ..., 0.        , 1.        ,
        0.01670844],
       [0.        , 0.        , 3.        , ..., 0.        , 0.        ,
        0.0726817 ],
       [0.        , 0.        , 3.        , ..., 0.        , 0.        ,
        0.02389307]])

## 5.7 Transforming ytest and ytrain

### Price conversion into log1p
- For Better Scaling Numerical Values

In [50]:
# log1p
ytrain_t = np.log1p(ytrain)
ytest_t = np.log1p(ytest)

## 5.8 Training Models on xtrain and ytrain

In [51]:
results = []

for name, model in models.items():

  # Train
  model.fit(xtrain_t, ytrain_t)

  # Predict
  y_pred = model.predict(xtest_t)

  # Metrics
  mae = mean_absolute_error(ytest_t, y_pred)
  r2 = r2_score(ytest_t, y_pred)

  # Storing the Results
  results.append({
      "Model" :name,
      "MAE" : mae,
      "R2 Score": r2
  })

  print("="*60)
  print(f"\nModel : {name}\n")
  print(f"MAE : {mae}")
  print(f"R2 Score: {r2*100:.2f}%")
  print("="*60)


Model : Linear Regression

MAE : 0.36331691740098637
R2 Score: 68.52%

Model : Random Forest

MAE : 0.25486857744364205
R2 Score: 82.29%

Model : Gradient Boosting

MAE : 0.28460542435943237
R2 Score: 80.62%

Model : XGBoost

MAE : 0.2648320787076904
R2 Score: 82.67%


## 5.9 Comparision of Loaded Models

In [52]:
# Results dataframe

results_df = pd.DataFrame(results)
results_df.sort_values(by = "R2 Score", ascending = False)

,Model,MAE,R2 Score
3,XGBoost,0.264832,0.826735
1,Random Forest,0.254869,0.822919
2,Gradient Boosting,0.284605,0.806224
0,Linear Regression,0.363317,0.685163


## 5.10 Hyperparameters Tunning

- Random Forest
- XGBoost

### 1. Random Forest

In [53]:
# Parameter Grid

rf_params = {
    "n_estimators": [100,200,300,500], # number of decision trees in the forest
    "max_depth" : [5, 10, 20, 30], #  maximum depth of each decision tree
    "min_samples_split" : [2,5, 10], # minimum number of samples required to split an internal node
    "min_samples_leaf" : [1,2,4], # minimum number of samples required at a leaf node
    "max_features": ["sqrt", "log2"] #features considered at each split
}

In [54]:
# Randomized Search

rf = RandomForestRegressor(random_state = 42)

rf_random = RandomizedSearchCV(
    estimator = rf,
    param_distributions = rf_params,
    n_iter = 20,
    cv = 5,
    scoring = "neg_mean_absolute_error",
    random_state = 42,
    n_jobs = -1
)

rf_random.fit(xtrain_t, ytrain_t)

RandomizedSearchCV(cv=5, estimator=RandomForestRegressor(random_state=42),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'max_depth': [5, 10, 20, 30],
                                        'max_features': ['sqrt', 'log2'],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [100, 200, 300, 500]},
                   random_state=42, scoring='neg_mean_absolute_error')

In [55]:
# Metrics

best_rf = rf_random.best_estimator_

y_pred_rf = best_rf.predict(xtest_t)

mae_furnished=  mean_absolute_error(ytest_t, y_pred_rf)

print("Best RF MAE : ",mae_furnished)
print("Best RF R2 Score: ", r2_score(ytest_t, y_pred_rf))

Best RF MAE :  0.2606259870670155
Best RF R2 Score:  0.8253158957137712


### 2. XGBoost

In [56]:
xgb_params = {
    "n_estimators": [100,200,300,500],
    "max_depth" : [3, 5, 7, 10],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree" : [0.7, 0.8, 0.9, 1.0]
}

In [57]:
# Randomized Search
xgb = XGBRegressor(random_state = 42, verbosity = 0)

xgb_random = RandomizedSearchCV(
    estimator = xgb,
    param_distributions = xgb_params,
    n_iter = 20,
    cv = 5,
    scoring = "neg_mean_absolute_error",
    random_state = 42,
    n_jobs = -1
)

xgb_random.fit(xtrain_t, ytrain_t)

RandomizedSearchCV(cv=5,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          feature_weights=None, gamma=None,
                                          grow_policy=None,
                                          importance_type=None,
                                          interaction_constraint...
                                          monotone_constraints=None,
                                          multi_strategy=None,
                                          n_estimators=None, n_jobs=None,
                                          num_parallel_tree=None, ...),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.7, 0.8, 0.9,
                                                             1.0],
                                        'learning_rate': [0.01, 0.05, 0.1, 0.2],
                                        'max_depth': [3, 5, 7, 10],
                                        'n_estimators': [100, 200, 300, 500],
                                        'subsample': [0.7, 0.8, 0.9, 1.0]},
                   random_state=42, scoring='neg_mean_absolute_error')

In [58]:
# Metrics
best_xgb = xgb_random.best_estimator_

y_pred_xgb = best_xgb.predict(xtest_t)

mae_furnished=  mean_absolute_error(ytest_t, y_pred_xgb)

print("Best RF MAE : ",mae_furnished)
print("Best XGB R2: ", r2_score(ytest_t, y_pred_xgb))

Best RF MAE :  0.25704162279148085
Best XGB R2:  0.8305855294099285


## 5.11 Selecting the Best Performance Model

## 5.12 Saving Tunned Model

In [59]:
#Saving the Models

joblib.dump(transformer, "furnished_transformer.pkl")
joblib.dump(rf, "rf_furnished_tunned.pkl")

['rf_furnished_tunned.pkl']

## 5.13 Saving mae of unfurnished to json

In [60]:
metrics ={
    "mae": mae_furnished
}

with open("furnished_mae.json", "w") as f:
  json.dump(metrics, f)

## 5.14 Full Production Pipeline


In [61]:
full_production_pipeline = Pipeline(
    steps=[
        ("preprocessing", transformer),
        ("model", best_rf)
    ]
)

## 5.15 Saving Full Production pipeline

In [62]:
joblib.dump(full_production_pipeline, "furnished_full_production_pipeline.pkl")

['furnished_full_production_pipeline.pkl']

## 5.16 Testing the model

In [63]:
# User Inputs
user = {
    "balconies" : 1.0,
    "bathroom" : 2.0,
    "house type" : "New",
    "no. of additional rooms" : 0.0,
    "total area" : 1200.0,
    "total rooms" : 3.0,
     "Car Parking" :"Yes",
    "Power Backup": "No",
    "AC": "Yes",
    "TV" : "No",
    "Refrigerator" : "No",
    "Sofa": "No",
    "Wardrobe": "Yes",
    "Washing Machine": "No",
    "Gas connection": "Yes",
    "BED": "No"
}

# Converting to dataframe
user = pd.DataFrame([user])
user

,balconies,bathroom,house type,no. of additional rooms,total area,total rooms,Car Parking,Power Backup,AC,TV,Refrigerator,Sofa,Wardrobe,Washing Machine,Gas connection,BED
0,1.0,2.0,New,0.0,1200.0,3.0,Yes,No,Yes,No,No,No,Yes,No,Yes,No


In [64]:
# Loading the Full production Model
model = joblib.load("furnished_full_production_pipeline.pkl")

In [65]:
# Predicting the Prices
model.predict(user)

array([15.71840986])

In [66]:
print("The Price Prediction is : ", np.exp(model.predict(user)))

The Price Prediction is :  [6705307.02732525]


## Saving Clean Dataset

In [67]:
data.to_csv("pune_property_cleaned_Nan_filled.csv", index=False)